In [1]:
#LetNet5经典CNN的实现
import math

import dnnlpy
import dnnlpy.nn as dnn
import torch
import torch.nn as nn
import torchinfo
from torch import Tensor

print('PyTorch version:', torch.__version__)

PyTorch version: 2.14.0


In [ ]:
#经典LeNet5
class LeNet5(nn.Module):
    """A practical implementation of the classic LeNet-5 architecture."""

    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            dnn.Conv2d(1, 6, kernel_size=5),
            dnn.Tanh(),
            dnn.AvgPool2d(kernel_size=2),
            dnn.Conv2d(6, 16, kernel_size=5),
            dnn.Tanh(),
            dnn.AvgPool2d(kernel_size=2),
            dnn.Conv2d(16, 120, kernel_size=5),
            dnn.Tanh(),
        )
        self.classifier = nn.Sequential(
            dnn.Flatten(),
            dnn.Linear(120, 84),
            dnn.Tanh(),
            dnn.Linear(84, num_classes),
        )

    def forward(self, x: Tensor) -> Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x


model = LeNet5(num_classes=10)
x = torch.randn(8, 1, 32, 32)
logits = model(x)

print(model, end='\n\n')
print('Input shape:', x.shape)
print('Output shape:', logits.shape)

In [2]:
#现代LetNet5：
# 用 ReLU 替代 Tanh；
#用 max pooling 或 stride convolution 替代 average pooling
#使用 padding，使卷积前后的空间尺寸更容易控制；
#使用 adaptive pooling/GAP，避免分类头依赖固定输入分辨率；
#根据需要加入 normalization 和 dropout
class ModernLeNet5(nn.Module):
    """A modernized LeNet-style CNN with adaptive pooling."""

    def __init__(self, num_classes: int = 10) -> None:
        super().__init__()
        self.features = nn.Sequential(
            dnn.Conv2d(1, 16, kernel_size=3, padding=1),
            dnn.ReLU(),
            dnn.MaxPool2d(kernel_size=2),
            dnn.Conv2d(16, 32, kernel_size=3, padding=1),
            dnn.ReLU(),
            dnn.MaxPool2d(kernel_size=2),
        )
        self.pool = dnn.AdaptiveAvgPool2d(1)
        self.flatten = dnn.Flatten()
        self.classifier = dnn.Linear(32, num_classes)

    def forward(self, x: Tensor) -> Tensor:
        x = self.features(x)
        x = self.pool(x)
        x = self.flatten(x)
        x = self.classifier(x)
        return x


model = ModernLeNet5(num_classes=10)
summary = torchinfo.summary(model, input_size=(8, 1, 32, 32))
print(summary)

Layer (type:depth-idx)                   Output Shape              Param #
ModernLeNet5                             [8, 10]                   --
├─Sequential: 1-1                        [8, 32, 8, 8]             --
│    └─Conv2d: 2-1                       [8, 16, 32, 32]           160
│    └─ReLU: 2-2                         [8, 16, 32, 32]           --
│    └─MaxPool2d: 2-3                    [8, 16, 16, 16]           --
│    └─Conv2d: 2-4                       [8, 32, 16, 16]           4,640
│    └─ReLU: 2-5                         [8, 32, 16, 16]           --
│    └─MaxPool2d: 2-6                    [8, 32, 8, 8]             --
├─AdaptiveAvgPool2d: 1-2                 [8, 32, 1, 1]             --
├─Flatten: 1-3                           [8, 32]                   --
├─Linear: 1-4                            [8, 10]                   330
Total params: 5,130
Trainable params: 5,130
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 10.82
Input size (MB): 0.03
Forward/backward pa